In [1]:
TEST_X = "f24-38615-hw3/test_X.csv"
TRAIN_X = "f24-38615-hw3/train_X.csv"
TRAIN_Y = "f24-38615-hw3/train_y.csv"

In [2]:
import pandas as pd

In [3]:
x = pd.read_csv(TRAIN_X).drop(columns=['Unnamed: 0'])
# x = pd.read_csv(TRAIN_X)
# x.describe()

In [4]:
y = pd.read_csv(TRAIN_Y).drop(columns=['Unnamed: 0'])
# y = pd.read_csv(TRAIN_Y)
y

,xml_neoplasm_histologic_grade
0,0
1,1
2,0
3,0
4,1
...,...
439,1
440,1
441,1
442,0


In [5]:
# tto_y = pd.read_csv(TRAIN_Y)
tt_x = pd.read_csv(TEST_X).drop(columns=['Unnamed: 0'])
tt_x

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,ENSG00000001460,...,ENSG00000282651,ENSG00000282815,ENSG00000282939,ENSG00000283063,ENSG00000283439,ENSG00000283463,ENSG00000283526,ENSG00000283586,ENSG00000283632,ENSG00000283697
0,7.503849,1.685440,8.090089,5.546588,3.412025,5.871539,8.395029,5.894996,6.313896,4.286261,...,3.626396,-2.640463,-7.278265,0.721287,-7.278265,4.672536,5.153675,-7.278265,4.843800,4.302864
1,6.112919,-5.425877,5.604743,4.248246,3.374033,3.370075,6.804015,4.646287,6.236134,4.167541,...,-5.425877,-2.143410,1.956699,1.377768,-5.425877,4.560829,3.735085,-5.425877,4.105789,4.803357
2,6.183846,1.217355,6.093903,4.403216,5.722867,6.123209,7.680258,4.983386,5.626569,3.518791,...,11.317818,-5.259782,5.093316,3.800607,-5.259782,4.966710,2.829487,-5.259782,5.118704,5.024979
3,6.325535,-0.197432,6.722632,4.509093,4.941256,3.700171,7.492606,5.031053,5.859242,4.252114,...,3.536141,-1.362703,3.480703,0.776383,-0.999629,4.794226,-5.399485,-5.399485,4.660878,4.684343
4,7.162383,-5.539710,6.186110,5.270282,5.412103,3.781568,7.285779,4.847552,6.337205,5.622222,...,5.112772,-5.539710,4.369268,2.927849,0.791397,4.542333,4.628775,-5.539710,4.812787,4.524834
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,6.949029,2.192625,7.613706,5.131517,4.292879,5.337764,7.668408,5.159289,5.645662,3.082916,...,5.515318,-5.785898,4.229114,2.875184,-0.303496,3.322246,3.683199,-5.785898,4.730559,3.251810
106,7.419322,0.472349,6.723539,5.176782,4.297442,4.591905,7.171547,6.693665,5.782247,5.965700,...,2.737453,-1.781791,3.509502,0.021815,-6.758708,4.684549,4.795272,-6.758708,4.234056,4.614238
107,7.135800,0.479097,7.051451,6.042828,4.523706,7.250545,7.577726,5.622388,6.447874,4.835028,...,5.951147,-5.960456,4.025253,2.785925,-5.960456,4.236003,5.597498,-5.960456,5.279498,3.813894
108,6.859635,2.159767,7.254992,4.900402,4.256251,6.503227,6.810730,6.036136,6.282137,4.745579,...,4.462337,-5.553034,4.325258,2.593648,-5.553034,4.727639,2.380364,-5.553034,5.523681,4.088765


In [6]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.metrics import accuracy_score

In [7]:
# standardize

In [8]:
scaler = StandardScaler()

In [9]:
x_scaled = scaler.fit_transform(x)

In [10]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [11]:
models = {
    'Linear Regression': LinearRegression(),
    'Logistic Regression': LogisticRegression(max_iter=10000),
    'Ridge': Ridge(),
    'Lasso': Lasso()
}

In [12]:
alphas = {'alpha': [0.1, 1, 10, 100]}

In [13]:
ridge_search = GridSearchCV(Ridge(), alphas, cv=5)
lasso_search = GridSearchCV(Lasso(), alphas, cv=5)


In [14]:
for name, model in models.items():
    if name in ['Ridge', 'Lasso']:
        if name == 'Ridge':
            ridge_search.fit(x_train, y_train)
            best_model = ridge_search.best_estimator_
        else:
            lasso_search.fit(x_train, y_train)
            best_model = lasso_search.best_estimator_
        y_pred = best_model.predict(x_test)
        # print(f"Best {name} alpha: {best_model.alpha_}")
    else:
        model.fit(x_train, y_train)
        y_pred = model.predict(x_test)
    
    # Classification metrics
    y_pred_binary = [1 if i >= 0.5 else 0 for i in y_pred]
    print(f"\n{name} Classification Report:\n", classification_report(y_test, y_pred_binary))
    print(f"{name} ROC-AUC Score:", roc_auc_score(y_test, y_pred))


Linear Regression Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.81      0.81        36
           1       0.87      0.87      0.87        53

    accuracy                           0.84        89
   macro avg       0.84      0.84      0.84        89
weighted avg       0.84      0.84      0.84        89

Linear Regression ROC-AUC Score: 0.9051362683438154


/opt/homebrew/anaconda3/envs/CMU_ML/lib/python3.12/site-packages/sklearn/utils/validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)



Logistic Regression Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.81      0.82        36
           1       0.87      0.89      0.88        53

    accuracy                           0.85        89
   macro avg       0.85      0.85      0.85        89
weighted avg       0.85      0.85      0.85        89

Logistic Regression ROC-AUC Score: 0.8461740041928721

Ridge Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.81      0.81        36
           1       0.87      0.87      0.87        53

    accuracy                           0.84        89
   macro avg       0.84      0.84      0.84        89
weighted avg       0.84      0.84      0.84        89

Ridge ROC-AUC Score: 0.9051362683438154


KeyboardInterrupt: 

In [15]:
# Lasso (L1 regularization)
logreg_l1 = LogisticRegression(max_iter=10000)
logreg_l1.fit(x_train, y_train)
y_pred_l1 = logreg_l1.predict(x_test)
print("L1 Logistic Regression Accuracy: ", accuracy_score(y_test, y_pred_l1))

/opt/homebrew/anaconda3/envs/CMU_ML/lib/python3.12/site-packages/sklearn/utils/validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


L1 Logistic Regression Accuracy:  0.8539325842696629


In [16]:
new_test_x = tt_x

In [17]:
kaggle_submit_pred = logreg_l1.predict(tt_x)
kaggle_submit_pred

array([1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1,
       0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0,
       0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0,
       1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1,
       1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1])

In [18]:
pd.DataFrame(kaggle_submit_pred).to_csv('op.csv', index=False)

In [19]:
from sklearn.utils import shuffle
from sklearn.metrics import classification_report, roc_auc_score

# 1. Train the best model on the original dataset
best_model = Ridge(alpha=1)  # Assuming Ridge with alpha=1 was the best model from previous tuning
best_model.fit(x_train, y_train)

# # 2. Evaluate performance on the real test set (original labels)
y_pred = best_model.predict(x_test)
y_pred_binary = [1 if i >= 0.5 else 0 for i in y_pred]

print("Original Model Performance (Real Labels):")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred)}")
print(f"Classification Report:\n", classification_report(y_test, y_pred_binary))

# # 3. Shuffle the labels (y-Randomization)
Y_shuffled = shuffle(y, random_state=42)

# # 4. Train the model on shuffled labels
best_model.fit(x_scaled, Y_shuffled)

# # 5. Evaluate the model's performance on the test set with shuffled labels
y_pred_shuffled = best_model.predict(x_test)
y_pred_shuffled_binary = [1 if i >= 0.5 else 0 for i in y_pred_shuffled]

print("\nModel Performance with Shuffled Labels (y-Randomization):")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_shuffled)}")
print(f"Classification Report:\n", classification_report(y_test, y_pred_shuffled_binary))


Original Model Performance (Real Labels):
ROC-AUC Score: 0.9051362683438154
Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.81      0.81        36
           1       0.87      0.87      0.87        53

    accuracy                           0.84        89
   macro avg       0.84      0.84      0.84        89
weighted avg       0.84      0.84      0.84        89


Model Performance with Shuffled Labels (y-Randomization):
ROC-AUC Score: 0.39622641509433965
Classification Report:
               precision    recall  f1-score   support

           0       0.36      0.44      0.40        36
           1       0.55      0.45      0.49        53

    accuracy                           0.45        89
   macro avg       0.45      0.45      0.44        89
weighted avg       0.47      0.45      0.45        89



/opt/homebrew/anaconda3/envs/CMU_ML/lib/python3.12/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but Ridge was fitted without feature names
  warnings.warn(


In [20]:
import numpy as np
import matplotlib.pyplot as plt

# Assuming best_model is the model you want to analyze (e.g., Ridge, Lasso, or LogisticRegression)
model = best_model  # The model you've already trained, like Ridge or Lasso

# 1. Extract model coefficients
coefficients = model.coef_

# 2. Get the absolute values of the coefficients to understand their importance (magnitude)
abs_coefficients = np.abs(coefficients)

# 3. Sort the features by importance (descending order)
sorted_idx = np.argsort(abs_coefficients)[::-1]

# 4. Top 10 most important features
top_n = 10  # You can change this to display more or fewer features
top_features = sorted_idx[:top_n]

print("Top 10 Important Features (by index):", top_features)
print("Top 10 Coefficients:", coefficients[top_features])

# 5. Optionally, plot the top features and their coefficients
plt.figure(figsize=(10, 6))
plt.bar(range(top_n), coefficients[top_features], align='center')
plt.xticks(range(top_n), top_features, rotation=90)  # You can replace 'top_features' with feature names if available
plt.ylabel('Coefficient Value')
plt.title(f'Top {top_n} Most Important Features')
plt.show()


Top 10 Important Features (by index): [[ 2587  3898 16199 ... 14370 17401  9145]]


IndexError: index 2587 is out of bounds for axis 0 with size 1